# LLM にツール（関数）を渡して呼び出させ、実行結果を戻して答えさせる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/21_llm/llm_tool_calling.ipynb)

NLP クックブック 21-14 ／ 大規模言語モデル (LLM)

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

手元のオープンな LLM に Python 関数の定義（名前・説明・引数の型）を渡し、運賃照会や為替換算、営業日数の計算を必要に応じて呼び出させて、その実行結果をもとに最終的な回答を作らせたい。ツールを渡さない場合と比べ、数値の正確さや不要な呼び出しが起きないかを確かめる。

In [1]:
import datetime, inspect, json, re, time, unicodedata
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("Colab のランタイムを GPU に変更してください")
torch.manual_seed(0)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere 以降。T4 では float16 を使う
DTYPE = torch.bfloat16 if BF16 else torch.float16
name = "Qwen/Qwen2.5-1.5B-Instruct"                   # Apache 2.0。tools 引数に対応したチャットテンプレートを持つ
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype=DTYPE, device_map="auto").eval()

# ツールが参照する固定の運賃表・為替レート表（外部データの代わりに辞書で用意する）
FARES = {
    ("東京", "名古屋"): {"自由席": 10560, "指定席": 11300, "グリーン車": 14960, "所要時間_分": 98},
    ("東京", "京都"): {"自由席": 13320, "指定席": 14170, "グリーン車": 19040, "所要時間_分": 133},
    ("東京", "新大阪"): {"自由席": 13870, "指定席": 14720, "グリーン車": 19590, "所要時間_分": 148},
    ("東京", "仙台"): {"自由席": 10560, "指定席": 11410, "グリーン車": 15070, "所要時間_分": 91},
    ("東京", "金沢"): {"自由席": 13850, "指定席": 14380, "グリーン車": 19450, "所要時間_分": 148},
    ("新大阪", "博多"): {"自由席": 14750, "指定席": 15600, "グリーン車": 21700, "所要時間_分": 148},
}
RATES_TO_JPY = {"JPY": 1.0, "USD": 152.5, "EUR": 164.0, "GBP": 193.5, "KRW": 0.11}

def get_train_fare(origin: str, destination: str, seat_type: str = "指定席") -> dict:
    """主要区間の新幹線運賃（円）と所要時間（分）を登録表から調べる。

    Args:
        origin: 出発駅名（例: "東京"）
        destination: 到着駅名（例: "名古屋"）
        seat_type: 座席種別（"自由席" / "指定席" / "グリーン車"）
    """
    o, d = origin.strip().removesuffix("駅"), destination.strip().removesuffix("駅")
    key = (o, d) if (o, d) in FARES else (d, o)
    if key not in FARES:
        return {"error": f"区間 {origin}->{destination} は登録されていません"}
    if seat_type not in ("自由席", "指定席", "グリーン車"):
        return {"error": f"座席種別 '{seat_type}' は自由席・指定席・グリーン車から選んでください"}
    info = FARES[key]
    return {"origin": o, "destination": d, "seat_type": seat_type,
            "fare_jpy": info[seat_type], "duration_min": info["所要時間_分"]}

def convert_currency(amount: float, from_currency: str, to_currency: str = "JPY") -> dict:
    """固定換算表を使って金額を通貨換算する。対応通貨は USD, EUR, GBP, KRW, JPY。

    Args:
        amount: 換算する金額
        from_currency: 元の通貨コード（例: "USD"）
        to_currency: 換算先の通貨コード（例: "JPY"）
    """
    src, dst = from_currency.strip().upper(), to_currency.strip().upper()
    if src not in RATES_TO_JPY or dst not in RATES_TO_JPY:
        return {"error": f"未対応の通貨です: {from_currency} -> {to_currency}"}
    converted = round(float(amount) * RATES_TO_JPY[src] / RATES_TO_JPY[dst], 2)
    return {"amount": float(amount), "from_currency": src, "to_currency": dst, "converted": converted}

def count_business_days(start_date: str, end_date: str) -> dict:
    """2 つの日付（両端を含む）の総日数と、土日を除いた営業日数を数える。

    Args:
        start_date: 開始日（YYYY-MM-DD 形式）
        end_date: 終了日（YYYY-MM-DD 形式）
    """
    try:
        d1 = datetime.date.fromisoformat(start_date.strip())
        d2 = datetime.date.fromisoformat(end_date.strip())
    except ValueError:
        return {"error": f"日付の形式が不正です: {start_date}, {end_date}（YYYY-MM-DD を使用）"}
    if d1 > d2:
        return {"error": f"開始日 {start_date} が終了日 {end_date} より後です"}
    total = (d2 - d1).days + 1
    biz = sum(1 for k in range(total) if (d1 + datetime.timedelta(days=k)).weekday() < 5)
    return {"start_date": start_date, "end_date": end_date, "total_days": total, "business_days": biz}

TOOLS = [get_train_fare, convert_currency, count_business_days]
TOOL_MAP = {fn.__name__: fn for fn in TOOLS}

# 型ヒントと Google 形式の docstring から <tools> 内の JSON スキーマが自動生成されることを確認する
prompt_sample = tokenizer.apply_chat_template(
    [{"role": "user", "content": "東京から名古屋まで指定席で行く運賃は？"}],
    tools=TOOLS, add_generation_prompt=True, tokenize=False,
)
print(prompt_sample)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "get_train_fare", "description": "主要区間の新幹線運賃（円）と所要時間（分）を登録表から調べる。", "parameters": {"type": "object", "properties": {"origin": {"type": "string", "description": "出発駅名（例: \"東京\"）"}, "destination": {"type": "string", "description": "到着駅名（例: \"名古屋\"）"}, "seat_type": {"type": "string", "description": "座席種別（\"自由席\" / \"指定席\" / \"グリーン車\"）"}}, "required": ["origin", "destination"]}, "return": {"type": "object"}}}
{"type": "function", "function": {"name": "convert_currency", "description": "固定換算表を使って金額を通貨換算する。対応通貨は USD, EUR, GBP, KRW, JPY。", "parameters": {"type": "object", "properties": {"amount": {"type": "number", "description": "換算する金額"}, "from_currency": {"type": "string", "description": "元の通貨コード（例: \"USD

In [2]:
SYS_TOOL = (
    "提供されたツールを使って正確に調べて答えてください。"
    "前のツールの結果が次の呼び出しに必要なときは 1 回ずつ順に呼び出してください。"
    "ツールが不要な一般的な質問には直接答えてください。"
    "ツールが error を返した場合は数値を推測せず、登録がなく案内できないと答えてください。"
)

def parse_tool_calls(text):
    """<tool_call>...</tool_call> 内の JSON（name と arguments）を取り出し、成功件数と失敗件数も返す"""
    calls, n_ok, n_fail = [], 0, 0
    for block in re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", text, flags=re.DOTALL):
        try:
            obj = json.loads(block)
        except json.JSONDecodeError:
            n_fail += 1
            continue
        if isinstance(obj, dict) and isinstance(obj.get("name"), str) and isinstance(obj.get("arguments", {}), dict):
            calls.append({"name": obj["name"], "arguments": obj.get("arguments", {})})
            n_ok += 1
        else:
            n_fail += 1
    return calls, n_ok, n_fail

def execute_tool(name, args):
    """ホワイトリスト辞書 TOOL_MAP と型注釈を使って安全に引数変換・実行する（eval は使わない）"""
    if name not in TOOL_MAP:
        return {"error": f"未登録のツールです: {name}"}
    fn = TOOL_MAP[name]
    sig = inspect.signature(fn)
    cleaned = {}
    for k, v in args.items():
        if k not in sig.parameters:
            return {"error": f"不正な引数名です: {k}"}
        ann = sig.parameters[k].annotation
        if ann is float:
            if isinstance(v, bool):
                return {"error": f"引数 {k} に真偽値は使えません"}
            try:
                cleaned[k] = float(v)
            except (TypeError, ValueError):
                return {"error": f"引数 {k} は数値である必要があります"}
        elif ann is str:
            if not isinstance(v, str):
                return {"error": f"引数 {k} は文字列である必要があります"}
            cleaned[k] = v
        else:
            cleaned[k] = v
    try:
        return fn(**cleaned)
    except TypeError as e:
        return {"error": f"引数が不足または不正です: {e}"}

@torch.no_grad()
def run_with_tools(question, max_steps=4, max_new_tokens=200):
    messages = [{"role": "system", "content": SYS_TOOL}, {"role": "user", "content": question}]
    trace, ok_sum, fail_sum = [], 0, 0
    for step in range(1, max_steps + 1):
        inputs = tokenizer.apply_chat_template(
            messages, tools=TOOLS, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(model.device)
        # </tool_call> でいったん生成を止め、実行結果を返してから次の判断に進ませる
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            stop_strings=["</tool_call>"], tokenizer=tokenizer, pad_token_id=tokenizer.eos_token_id,
        )
        text = tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        calls, n_ok, n_fail = parse_tool_calls(text)
        ok_sum += n_ok
        fail_sum += n_fail
        if not calls:
            return text, trace, ok_sum, fail_sum
        messages.append({"role": "assistant", "tool_calls": [{"type": "function", "function": c} for c in calls]})
        for call in calls:
            res = execute_tool(call["name"], call["arguments"])
            trace.append({"step": step, "name": call["name"], "arguments": call["arguments"], "result": res})
            messages.append({"role": "tool", "name": call["name"], "content": json.dumps(res, ensure_ascii=False)})
    return text, trace, ok_sum, fail_sum

@torch.no_grad()
def answer_direct(question, max_new_tokens=200):
    """比較用: ツール定義を渡さずに直接答えさせる"""
    messages = [{"role": "system", "content": "簡潔かつ正確に答えてください。"}, {"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

demos = [
    ("1 回の呼び出し", "東京から名古屋まで指定席で行く場合の新幹線運賃と所要時間を教えてください。"),
    ("複数ステップ", "東京から名古屋までの自由席と、東京から京都までの指定席の運賃をそれぞれ調べ、合計いくらになるか教えてください。"),
    ("未登録エラー", "東京から札幌まで指定席で行く場合の新幹線運賃を教えてください。"),
    ("ツール不要", "Python でリストから重複要素を取り除くにはどのデータ型を使いますか。一言で教えてください。"),
]
for label, q in demos:
    ans, tr, _, _ = run_with_tools(q)
    print(f"[{label}] Q: {q}")
    for t in tr:
        print(f"  step {t['step']}: {t['name']}({t['arguments']}) -> {t['result']}")
    print("  応答:", ans.replace("\n", " "))

[1 回の呼び出し] Q: 東京から名古屋まで指定席で行く場合の新幹線運賃と所要時間を教えてください。
  step 1: get_train_fare({'origin': '東京', 'destination': '名古屋', 'seat_type': '指定席'}) -> {'origin': '東京', 'destination': '名古屋', 'seat_type': '指定席', 'fare_jpy': 11300, 'duration_min': 98}
  応答: 東京から名古屋への指定席での新幹線運賃は約1万1千300円です。所要時間は約98分です。
[複数ステップ] Q: 東京から名古屋までの自由席と、東京から京都までの指定席の運賃をそれぞれ調べ、合計いくらになるか教えてください。
  step 1: get_train_fare({'origin': '東京', 'destination': '名古屋', 'seat_type': '自由席'}) -> {'origin': '東京', 'destination': '名古屋', 'seat_type': '自由席', 'fare_jpy': 10560, 'duration_min': 98}
  step 2: get_train_fare({'origin': '東京', 'destination': '京都', 'seat_type': '指定席'}) -> {'origin': '東京', 'destination': '京都', 'seat_type': '指定席', 'fare_jpy': 14170, 'duration_min': 133}
  応答: 東京から名古屋までの自由席の運賃は10,560円で、所要時間は約98分です。 東京から京都までの指定席の運賃は14,170円で、所要時間は約133分です。 合計は24,730円になります。
[未登録エラー] Q: 東京から札幌まで指定席で行く場合の新幹線運賃を教えてください。
  step 1: get_train_fare({'origin': '東京', 'destination': '札幌', 'seat_type': '指定席'}) -> {'error': '区間 東京->札幌 は登録されていません'}
  応答: 

In [3]:
# 定量評価: ツール必須 24 問（運賃 8・為替 8・営業日 8）＋ ツール不要 6 問（計 30 問）
questions = [
    ("運賃", "get_train_fare", "東京から名古屋まで指定席で行く場合の新幹線運賃（円）と所要時間（分）を教えてください。", ["11300", "98"]),
    ("運賃", "get_train_fare", "東京から新大阪まで自由席で行く場合の新幹線運賃と所要時間を教えてください。", ["13870", "148"]),
    ("運賃", "get_train_fare", "新大阪から博多までグリーン車で行く場合の新幹線運賃と所要時間を教えてください。", ["21700", "148"]),
    ("運賃", "get_train_fare", "東京から金沢まで指定席で行く場合の新幹線運賃と所要時間を教えてください。", ["14380", "148"]),
    ("運賃", "get_train_fare", "東京から名古屋までの自由席と、東京から京都までの指定席の運賃をそれぞれ調べ、合計は何円か教えてください。", ["24730"]),
    ("運賃", "get_train_fare", "東京から仙台までの指定席と、東京から名古屋までの指定席の運賃をそれぞれ調べ、差額は何円か教えてください。", ["110"]),
    ("運賃", "get_train_fare", "東京から札幌まで指定席で行く場合の新幹線運賃を教えてください。", ["未登録"]),
    ("運賃", "get_train_fare", "名古屋から博多まで指定席で行く場合の新幹線運賃を教えてください。", ["未登録"]),
    ("為替", "convert_currency", "200 USD を JPY に換算するといくらになりますか。", ["30500"]),
    ("為替", "convert_currency", "150 EUR を JPY に換算するといくらになりますか。", ["24600"]),
    ("為替", "convert_currency", "80 GBP を JPY に換算するといくらになりますか。", ["15480"]),
    ("為替", "convert_currency", "50000 KRW を JPY に換算するといくらになりますか。", ["5500"]),
    ("為替", "convert_currency", "30500 JPY を USD に換算するといくらになりますか。", ["200"]),
    ("為替", "convert_currency", "100 EUR を USD に換算するといくらになりますか。", ["107.54"]),
    ("為替", "convert_currency", "100 USD と 50 EUR をそれぞれ JPY に換算すると合計は何円ですか。", ["23450"]),
    ("為替", "convert_currency", "200 GBP と 100 USD をそれぞれ JPY に換算し、両者の差額は何円か教えてください。", ["23450"]),
    ("営業日", "count_business_days", "2025-05-01 から 2025-05-10 までの総日数と土日を除いた営業日数を教えてください。", ["10", "7"]),
    ("営業日", "count_business_days", "2025-04-07 から 2025-04-11 までの総日数と営業日数を教えてください。", ["5"]),
    ("営業日", "count_business_days", "2025-06-02 から 2025-06-15 までの総日数と営業日数を教えてください。", ["14", "10"]),
    ("営業日", "count_business_days", "2025-07-01 から 2025-07-31 までの総日数と営業日数を教えてください。", ["31", "23"]),
    ("営業日", "count_business_days", "2025-08-09 から 2025-08-17 までの総日数と営業日数を教えてください。", ["9", "5"]),
    ("営業日", "count_business_days", "2025-09-01 から 2025-09-12 までの総日数と営業日数を教えてください。", ["12", "10"]),
    ("営業日", "count_business_days", "2025-04-01 から 2025-04-14 までと、2025-04-15 から 2025-04-28 までの営業日数をそれぞれ教えてください。", ["10"]),
    ("営業日", "count_business_days", "2025-10-01 から 2025-10-15 までの営業日数を教えてください。", ["11"]),
    ("一般", None, "Python でリストから重複要素を取り除くにはどのデータ型を使いますか。", ["set", "セット"]),
    ("一般", None, "HTTP ステータスコード 404 は何を表しますか。", ["見つから", "not found", "存在しない"]),
    ("一般", None, "水の分子を表す化学式を教えてください。", ["h2o"]),
    ("一般", None, "TCP と UDP の主な違いを 1 文で説明してください。", ["接続", "信頼", "コネクション", "確実", "再送"]),
    ("一般", None, "JSON 形式で文字列を囲むのに使う引用符はシングルクォートとダブルクォートのどちらですか。", ["ダブル", '"']),
    ("一般", None, "1 バイトは何ビットですか。", ["8"]),
]

def normalize_text(s):
    s = unicodedata.normalize("NFKC", s).replace(",", "")
    return re.sub(r"(\d+(?:\.\d+)?)万(\d*)", lambda m: str(round(float(m.group(1)) * 10000) + int(m.group(2) or 0)), s)

def is_correct(kind, ans, keys):
    s = normalize_text(ans)
    if kind == "運賃" and keys == ["未登録"]:
        return ("登録" in s or "案内でき" in s) and not re.search(r"\d{4,}円", s)
    if kind == "一般":
        return any(k in s.lower() for k in keys)
    return all(k in s for k in keys)

torch.cuda.reset_peak_memory_stats()
records = {}
for mode in ["ツールなし", "ツールあり"]:
    t0, rows = time.time(), []
    for kind, exp_tool, q, keys in questions:
        if mode == "ツールなし":
            ans, tr, ok, fail = answer_direct(q), [], 0, 0
        else:
            ans, tr, ok, fail = run_with_tools(q)
        tools_used = [t["name"] for t in tr]
        rows.append({
            "種別": kind, "質問": q, "正解": is_correct(kind, ans, keys),
            "ツール選択一致": (exp_tool in tools_used) if exp_tool else (len(tools_used) == 0),
            "呼び出し回数": len(tr), "パース成功": ok, "パース失敗": fail,
            "呼び出し履歴": tools_used, "応答": ans,
        })
    elapsed = time.time() - t0
    df = pd.DataFrame(rows)
    records[mode] = df
    req, gen = df[df["種別"] != "一般"], df[df["種別"] == "一般"]
    n_parse = req["パース成功"].sum() + req["パース失敗"].sum()
    print(f"[{mode}] 所要 {elapsed:.1f} 秒 | ツール必須 24 問正解率 {req['正解'].mean():.3f} ({req['正解'].sum()}/24) | "
          f"ツール選択 {req['ツール選択一致'].mean():.3f} | 複数回呼び出し {(req['呼び出し回数'] >= 2).mean():.3f} | "
          f"JSON パース {req['パース成功'].sum()}/{n_parse or 1}")
    print(f"             ツール不要 6 問: 正解率 {gen['正解'].mean():.3f} ({gen['正解'].sum()}/6) | "
          f"不要な誤発火 {(gen['呼び出し回数'] > 0).mean():.3f}")

# 対応のあるブートストラップでツール必須 24 問の正解率差の 95% 信頼区間を求める
x = records["ツールあり"].loc[records["ツールあり"]["種別"] != "一般", "正解"].to_numpy(float)
y = records["ツールなし"].loc[records["ツールなし"]["種別"] != "一般", "正解"].to_numpy(float)
idx = np.random.default_rng(0).integers(0, len(x), (2000, len(x)))
boot = x[idx].mean(1) - y[idx].mean(1)
print(f"正解率の差（ツールあり − ツールなし）: {(x.mean() - y.mean()) * 100:+.1f} ポイント "
      f"[{np.percentile(boot, 2.5) * 100:+.1f}, {np.percentile(boot, 97.5) * 100:+.1f}]")

print("\n--- ツールありでの失敗例 ---")
for _, r in records["ツールあり"].query("not 正解").iterrows():
    print(f"  [{r['種別']}] {r['質問'][:34]}... | 呼出={r['呼び出し履歴']} | {r['応答'][:75].replace(chr(10), ' ')}")
print("\n--- ツールなしとの対比（成功例） ---")
for i in [0, 8, 16]:
    a = records["ツールあり"].iloc[i]
    b = records["ツールなし"].iloc[i]
    print(f"  Q: {a['質問'][:36]}...")
    print(f"    ツールなし: {b['応答'][:70].replace(chr(10), ' ')}")
    print(f"    ツールあり: {a['応答'][:70].replace(chr(10), ' ')}")
print(f"\nGPU メモリの最大使用量: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

[ツールなし] 所要 56.0 秒 | ツール必須 24 問正解率 0.083 (2/24) | ツール選択 0.000 | 複数回呼び出し 0.000 | JSON パース 0/1
             ツール不要 6 問: 正解率 1.000 (6/6) | 不要な誤発火 0.000
[ツールあり] 所要 99.5 秒 | ツール必須 24 問正解率 0.833 (20/24) | ツール選択 1.000 | 複数回呼び出し 0.125 | JSON パース 27/27
             ツール不要 6 問: 正解率 1.000 (6/6) | 不要な誤発火 0.000
正解率の差（ツールあり − ツールなし）: +75.0 ポイント [+58.3, +91.7]

--- ツールありでの失敗例 ---
  [運賃] 東京から新大阪まで自由席で行く場合の新幹線運賃と所要時間を教えてくだ... | 呼出=['get_train_fare'] | 東京から新大阪までの自由席での新幹線運賃は約13,870円です。所要時間は約1時間48分です。
  [運賃] 新大阪から博多までグリーン車で行く場合の新幹線運賃と所要時間を教えて... | 呼出=['get_train_fare'] | 新大阪から博多までのグリーン車の新幹線運賃は約2万1700円です。所要時間は約1時間48分です。
  [運賃] 東京から金沢まで指定席で行く場合の新幹線運賃と所要時間を教えてくださ... | 呼出=['get_train_fare'] | 東京から金沢への指定席での新幹線運賃は約14,380円です。所要時間は約1時間48分です。
  [運賃] 東京から仙台までの指定席と、東京から名古屋までの指定席の運賃をそれぞ... | 呼出=['get_train_fare'] | 東京から仙台への指定席の運賃は11,410円です。  東京から名古屋への指定席の運賃は約13,560円です。  差額は約2,150円です。

--- ツールなしとの対比（成功例） ---
  Q: 東京から名古屋まで指定席で行く場合の新幹線運賃（円）と所要時間（分）を教...
    ツールなし: 申し訳ありませんが、具体的な指定席や運賃料金等の詳細情報は提供できません。一般的な新幹線の運賃と所要時間については、

- ツール必須 24 問の正解率は、ツールなし 0.083 (2/24) からツールあり 0.833 (20/24) へ上がり、差は +75.0 ポイント [+58.3, +91.7]（対応のあるブートストラップによる 95% 信頼区間）だった。ツールなしでは登録表の運賃と独自レートの為替換算を 1 問も当てられず (0/16)、営業日数も両端を含む数え方や曜日判定を外して 2/8 にとどまった。
- ツールありでは 24 問すべてで期待したツールが選ばれ（選択率 1.000）、`<tool_call>` 内の JSON 引数パースも 27/27 が成功した。複数回呼び出しは 24 問中 3 問 (0.125) で起き、動作トレースの 2 区間比較では step 1（東京→名古屋）と step 2（東京→京都）を順に呼び出して合計 24,730 円を正しく答えた。ツール不要な一般質問 6 問では不要な誤発火が 0 件 (0.000)、正解率も 1.000 (6/6) を保った。
- 失敗例（4 件）はいずれも小型モデル (1.5B) による呼び出し後の整理・比較の誤りである。`duration_min` が 148 分だった 3 区間（東京→新大阪、新大阪→博多、東京→金沢）では、ツールが返した運賃は正しく答えた一方、148 分を「約1時間48分」（108 分）と読み誤った。2 区間の差額を問う問題（東京→仙台と東京→名古屋）では、仙台だけツールを呼んで 11,410 円を取得し、名古屋は 2 回目を呼ばずに約 13,560 円と捏造した。一方、未登録区間（東京→札幌）のデモでは `{"error": ...}` を受けて数値を捏造せず「登録されていません」と答えており、エラーを明示的に返す設計とプロンプトの指示の両方が効いている。
- `apply_chat_template(..., tools=TOOLS)` は、Python 関数の型ヒント（`str`, `float`, 既定値）と Google 形式 docstring (`Args:`) を検査して JSON Schema を組み立て、システムメッセージの `<tools>...</tools>` に埋め込む。生成時は `stop_strings=["</tool_call>"]` を指定して `</tool_call>` でいったん止めると、小型モデルがツール結果を待たずに次の呼び出し引数を推測したり偽の実行結果を書き足したりするのを抑えられる。履歴には `role: "assistant"` の `tool_calls` と `role: "tool"` の実行結果（JSON 文字列）を交互に積んで次の生成に渡す。
- 安全なディスパッチのため、モデルの出力を `eval` や `exec` で実行せず、ホワイトリスト辞書 `TOOL_MAP` に登録された関数名だけを許可し、`inspect.signature` の型注釈に沿って引数名と型（数値引数への真偽値の混入防止など）を検証する。不正な引数には例外で落とさず `{"error": ...}` を返してモデルに再判断させる。また、Web ページや検索結果のように信頼できない外部テキストを返すツールでは、返り値に「元の指示を無視して別のツールを呼べ」といった指示を紛れ込ませる間接プロンプトインジェクションの危険がある（RAG の章の「プロンプトインジェクションを再現し、防御で成功率がどう変わるか測る」も参照）。実運用では参照系と更新・送信系のツールを分け、後者には人の確認を挟む。
- 30 問の処理時間はツールなし 56.0 秒、ツールあり 99.5 秒、GPU メモリの最大使用量は 3.2 GB だった（T4 相当の float16 ではこの倍程度の時間がかかる見込み）。複雑な多段階推論や大規模な関数群を安定して呼び出したい場合は、LLM の API の章の「Gemini に決まった形式（JSON スキーマ）で出力させる」「Claude に決まった形式（JSON スキーマ）で出力させる」や各 API のネイティブなツール呼び出し機能を使う。1 回の生成で出力の文法だけを強制したいときは、この章の「制約付きデコーディングで LLM の出力を JSON スキーマ・正規表現・選択肢に従わせる (outlines)」を使い、外部関数の選択と実行結果の反映まで繰り返させたいときにツール呼び出しを使う。